==== VQCC v6 GENERATOR  |  ONE PASTE, ONE CELL =========================

This cell GENERATES /kaggle/working/submission.zip - the archive that was
uploaded to the competition.  It does not describe it; it rebuilds it.

  1. (optional) Add Input -> Datasets -> manwithacat/vqcc-apache-public-circuits
  2. Paste this whole block into ONE code cell.
  3. Run All.  It takes a few minutes: the dqi circuit alone is 530 MB.

HOW IT GETS THE INPUTS
  If the dataset is attached, the cell copies the 18 members out of
  /kaggle/input.  If it is NOT attached, the cell downloads the same 18
  files from the public Kaggle dataset API.  Either way every byte is
  checked against the SHA-256 pins below, and the output archive is the
  same file.

THE CELL FAILS CLOSED
  If any of the 18 members is missing, unpinned, or hash-mismatched, it
  raises SystemExit and writes NOTHING.  A green run is the only way to
  produce an archive.

EXPECTED OUTPUT
  18 of 18 members OK, 0 bad
  archive sha256 : 44ef406039b7003367db18d59c6822668afa766544f9314b5cf8dd752b16f604
  archive check  : MATCH - byte-identical to the verified archive

Do not upload this notebook.  Upload only the generated submission.zip.

Provenance: every member is a circuit published by another participant and
redistributed here under the Apache-2.0 licence of the competition's public
dataset.  See NOTICE.md in this repository for the per-member attribution.
No verifier was modified.  No competition data is included in this
repository.


In [ ]:
import os, sys, json, glob, hashlib, shutil, zipfile, tempfile, subprocess
import urllib.request, urllib.error, urllib.parse, time

DATASET = "manwithacat/vqcc-apache-public-circuits"
DATASET_OWNER, DATASET_SLUG = DATASET.split("/")
DOWNLOAD_BASE = ("https://www.kaggle.com/api/v1/datasets/download/"
                 + DATASET_OWNER + "/" + DATASET_SLUG + "/")

INPUT_ROOT = os.environ.get("VQCC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("VQCC_WORK_DIR", "/kaggle/working")

RUN_OFFICIAL_VERIFIERS = False

LAYOUT = [
    ["invsqrt.kmb", "circuits/invsqrt/af4698478fba.kmb"],
    ["dqi.kmb", "circuits/dqi/748b01a7a076.kmb"],
    ["dfthcsos/submission.npz", "factorizations/a56ec354e8b8.npz"],
    ["dfthcsos/angle_k_0.kmb", "circuits/chemistry/0ed964465654/angle_k_0.kmb"],
    ["dfthcsos/angle_k_1.kmb", "circuits/chemistry/0ed964465654/angle_k_1.kmb"],
    ["dfthcsos/angle_k_2.kmb", "circuits/chemistry/0ed964465654/angle_k_2.kmb"],
    ["dfthcsos/angle_k_3.kmb", "circuits/chemistry/0ed964465654/angle_k_3.kmb"],
    ["dfthcsos/angle_k_4.kmb", "circuits/chemistry/0ed964465654/angle_k_4.kmb"],
    ["dfthcsos/angle_k_5.kmb", "circuits/chemistry/0ed964465654/angle_k_5.kmb"],
    ["dfthcsos/angle_k_6.kmb", "circuits/chemistry/0ed964465654/angle_k_6.kmb"],
    ["dfthcsos/angle_k_7.kmb", "circuits/chemistry/0ed964465654/angle_k_7.kmb"],
    ["dfthcsos/angle_k_8.kmb", "circuits/chemistry/0ed964465654/angle_k_8.kmb"],
    ["dfthcsos/angle_k_9.kmb", "circuits/chemistry/0ed964465654/angle_k_9.kmb"],
    ["dfthcsos/angle_k_10.kmb", "circuits/chemistry/0ed964465654/angle_k_10.kmb"],
    ["dfthcsos/angle_k_11.kmb", "circuits/chemistry/0ed964465654/angle_k_11.kmb"],
    ["dfthcsos/angle_k_12.kmb", "circuits/chemistry/0ed964465654/angle_k_12.kmb"],
    ["dfthcsos/angle_k_13.kmb", "circuits/chemistry/0ed964465654/angle_k_13.kmb"],
    ["dfthcsos/angle_k_14.kmb", "circuits/chemistry/0ed964465654/angle_k_14.kmb"],
]
EXPECTED = {
    "invsqrt.kmb": "af4698478fbad775a5a271690cf4c2d8783caf0b8d12048106d1973c4ac6862e",
    "dqi.kmb": "748b01a7a07662c3f7e389b844e313a3d55dfb3b3d993414e5ad3dfac6fe154c",
    "dfthcsos/submission.npz": "ddce9a5bd7d67969e9b7ba4266fe1ae4c8946344553a6dbb7e1355d657bfc397",
    "dfthcsos/angle_k_0.kmb": "c28fdb135638296e1ccbad85ad707baa139321ceef5abecdc6bda86dbfdb47d0",
    "dfthcsos/angle_k_1.kmb": "2196682be8711775e8817009ad9ba5225caaa72ae91947b0e35e99942cb1163c",
    "dfthcsos/angle_k_2.kmb": "37be1597c4f7c99ee1de85fb2f87a82d7b97caab2bff666b7b7937cf583c94bd",
    "dfthcsos/angle_k_3.kmb": "28af1e9c3afef751c42f734b332d6fa199a6471bf526941786d9592ea32a6151",
    "dfthcsos/angle_k_4.kmb": "471cd8153570113eb3c36ce32be9bfaca85f15a4213af628a5b151ee9b4d0f6e",
    "dfthcsos/angle_k_5.kmb": "a6b94471064c865605d30ff40adb05c7b920dd32e9401db2b0cdc7e1526acccd",
    "dfthcsos/angle_k_6.kmb": "1058048b0247dd1bc81a662ab97badf51ade79585f5b07eb412c57e63f203856",
    "dfthcsos/angle_k_7.kmb": "9e0b2182beeed5fb278207ab246f8b04184aaaec7aef4e0f6862f386a6f389ae",
    "dfthcsos/angle_k_8.kmb": "7f761462e0def965b8a83f14a823139a0cae95e924fbaef610b8183766fd8aa6",
    "dfthcsos/angle_k_9.kmb": "5298314defe51446f3735febc99302886bb2107cbf4dad7f9446aa8f49e51c50",
    "dfthcsos/angle_k_10.kmb": "6eb67bbdce1eeb5b65c3b8c9a3ba4976729f8bf845f1ea3450a64c15581039d2",
    "dfthcsos/angle_k_11.kmb": "71f078bc607d3aebf324e699fe3fe42d6744df541d34247b36a2096da455fe29",
    "dfthcsos/angle_k_12.kmb": "a5a74fa106eb4fac32e13f4e37248af2c08d865f1e0a7dcf60572013f4986e8a",
    "dfthcsos/angle_k_13.kmb": "605f81430a30417a40ec7eb2271cf175fa61930ef1ab74425acd8eca2160e622",
    "dfthcsos/angle_k_14.kmb": "4e7c0a3730cb5a15887f3e56ca71acde4d7248e690a78df832063dc5c5666c49",
}
VERIFIED_ARCHIVE_SHA256 = "44ef406039b7003367db18d59c6822668afa766544f9314b5cf8dd752b16f604"
VERIFIED_ARCHIVE_BYTES = 26156017
SIZES = {
    "circuits/invsqrt/af4698478fba.kmb": 479055,
    "circuits/dqi/748b01a7a076.kmb": 529710608,
    "factorizations/a56ec354e8b8.npz": 1863164,
    "circuits/chemistry/0ed964465654/angle_k_0.kmb": 396933,
    "circuits/chemistry/0ed964465654/angle_k_1.kmb": 394900,
    "circuits/chemistry/0ed964465654/angle_k_2.kmb": 399458,
    "circuits/chemistry/0ed964465654/angle_k_3.kmb": 392891,
    "circuits/chemistry/0ed964465654/angle_k_4.kmb": 358122,
    "circuits/chemistry/0ed964465654/angle_k_5.kmb": 356361,
    "circuits/chemistry/0ed964465654/angle_k_6.kmb": 357332,
    "circuits/chemistry/0ed964465654/angle_k_7.kmb": 348960,
    "circuits/chemistry/0ed964465654/angle_k_8.kmb": 355400,
    "circuits/chemistry/0ed964465654/angle_k_9.kmb": 359657,
    "circuits/chemistry/0ed964465654/angle_k_10.kmb": 358836,
    "circuits/chemistry/0ed964465654/angle_k_11.kmb": 357681,
    "circuits/chemistry/0ed964465654/angle_k_12.kmb": 358465,
    "circuits/chemistry/0ed964465654/angle_k_13.kmb": 341040,
    "circuits/chemistry/0ed964465654/angle_k_14.kmb": 355951,
}

def find_dataset_root():
    slug = DATASET.split("/")[-1]
    cands = []
    for base in (INPUT_ROOT,):
        cands += [os.path.join(base, "datasets", DATASET),
                  os.path.join(base, DATASET),
                  os.path.join(base, slug)]
    for c in cands:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, "circuits")) \
           and os.path.isdir(os.path.join(c, "factorizations")):
            return c
    for base in (INPUT_ROOT,):
        for hit in sorted(glob.glob(os.path.join(base, "**",
                                                 "circuits/invsqrt/53000387a86b.kmb"),
                                 recursive=True)):
            root = hit.split("/circuits/")[0]
            if os.path.exists(os.path.join(root, "factorizations",
                                           "37204439f717.npz")):
                return root
    return None


def download_one(rel, dest):
    """Fetch one file from the PUBLIC Kaggle dataset API.

    curl is tried first: urllib silently truncated the 530 MB dqi circuit by
    8.5 MB on the real download, curl pulled the exact 529,710,608 bytes.
    Both paths are size-checked, and the caller hash-checks the result.
    """
    url = DOWNLOAD_BASE + urllib.parse.quote(rel, safe="")
    want_bytes = SIZES.get(rel)
    tmp = dest + ".part"
    last = None
    for attempt in (1, 2, 3, 4):
        try:
            if shutil.which("curl"):
                rc = subprocess.call(
                    ["curl", "-sSL", "--fail", "--retry", "3",
                     "--retry-delay", "3", "--max-time", "1800",
                     "-o", tmp, url])
                if rc != 0:
                    raise IOError("curl exit code %d" % rc)
            else:
                req = urllib.request.Request(
                    url, headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(req, timeout=300) as r, \
                        open(tmp, "wb") as f:
                    shutil.copyfileobj(r, f, 1024 * 1024)
            got_bytes = os.path.getsize(tmp)
            if want_bytes and got_bytes != want_bytes:
                raise IOError("short read: got %d bytes, expected %d"
                              % (got_bytes, want_bytes))
            shutil.move(tmp, dest)
            return
        except Exception as exc:
            last = exc
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass
            time.sleep(4 * attempt)
    raise last


DS = find_dataset_root()
if DS is not None:
    print("dataset root :", DS, "(attached - no download needed)")
else:
    DS = os.path.join(WORK, "vqcc", "source")
    print("dataset root : none attached - downloading the 18 files now.")
    print("              538 MB total, the dqi circuit is 530 MB of it.")
    print("              This takes a few minutes. Let it run.")
    os.makedirs(DS, exist_ok=True)
    for _member, remote in LAYOUT:
        dest = os.path.join(DS, remote)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            continue
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        print("   fetching", remote, flush=True)
        try:
            download_one(remote, dest)
        except Exception as exc:
            raise SystemExit(
                "STOP: could not download " + remote + " (" + str(exc) + ").\n"
                "  Fix ONE of these in the Kaggle UI:\n"
                "   (a) right-hand panel -> Session -> turn INTERNET ON, or\n"
                "   (b) right-hand panel -> Add Input -> Datasets -> search\n"
                "       'manwithacat/vqcc-apache-public-circuits' -> Add.\n"
                "  Then press Run All again.")

print("python       :", sys.version.split()[0])

CIRCUITS = os.path.join(WORK, "vqcc", "circuits")
if os.path.isdir(CIRCUITS):
    shutil.rmtree(CIRCUITS)
os.makedirs(CIRCUITS)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


rows, bad = [], 0
for member, remote in LAYOUT:
    src = os.path.join(DS, remote)
    dst = os.path.join(CIRCUITS, member.replace("/", os.sep))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(src):
        print("MISSING", remote)
        bad += 1
        continue
    shutil.copyfile(src, dst)
    got = sha256(dst)
    want = EXPECTED.get(member)
    if want is None:
        bad += 1
        os.remove(dst)
        print("%-5s %-30s no expected SHA-256 is pinned" % ("FAIL", member))
        continue
    ok = (got == want)
    if not ok:
        bad += 1
        os.remove(dst)
    rows.append({"member": member, "sha256": got,
                 "bytes": os.path.getsize(dst) if os.path.exists(dst) else 0,
                 "ok": ok, "expected": want, "upstream": remote})
    print("%-5s %-30s %s" % ("OK" if ok else "FAIL", member, got))

print()
print("%d of %d members OK, %d bad" % (len(rows) - bad, len(rows), bad))
if bad:
    raise SystemExit("STOP: a hash did not match. Nothing was packaged.")

OUT = os.path.join(WORK, "submission.zip")
FIXED = (2026, 11, 4, 0, 0, 0)

if os.path.exists(OUT):
    os.remove(OUT)
print("packing (the dqi circuit is 530 MB, so this is the slow step)...")
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
    for member, _remote in LAYOUT:
        path = os.path.join(CIRCUITS, member.replace("/", os.sep))
        info = zipfile.ZipInfo(member, date_time=FIXED)
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o644 << 16
        info.create_system = 3
        with open(path, "rb") as fh:
            z.writestr(info, fh.read(), compresslevel=9)

stage = tempfile.mkdtemp(prefix="vqcc_pack_")
with zipfile.ZipFile(OUT) as z:
    names = z.namelist()
    assert len(names) == 18, "expected 18 members, got %d" % len(names)
    z.extractall(stage)

manifest = {"members": []}
for member, _remote in LAYOUT:
    p = os.path.join(stage, member.replace("/", os.sep))
    assert os.path.exists(p), "member missing from the packed archive: " + member
    digest = sha256(p)
    want = EXPECTED[member]
    assert digest == want, "packed member does not match its pin: " + member
    manifest["members"].append({"member": member, "sha256": digest,
                                 "bytes": os.path.getsize(p)})
shutil.rmtree(stage, ignore_errors=True)

manifest["archive_sha256"] = sha256(OUT)
manifest["archive_bytes"] = os.path.getsize(OUT)
os.makedirs(os.path.join(WORK, "receipts"), exist_ok=True)
with open(os.path.join(WORK, "receipts", "submission-manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2)

print("archive        :", OUT)
print("archive bytes  :", manifest["archive_bytes"])
print("archive sha256 :", manifest["archive_sha256"])
print("members        :", len(manifest["members"]))
if manifest["archive_sha256"] == VERIFIED_ARCHIVE_SHA256:
    print("archive check  : MATCH - byte-identical to the verified archive")
else:
    print("archive check  : the 18 MEMBERS are pinned and identical; only the")
    print("                 zip compression bytes differ between zlib builds.")
    print("                 Upload this file.")

if RUN_OFFICIAL_VERIFIERS:
    try:
        VERIFIERS = None
        for root in (INPUT_ROOT, "input", "."):
            for hit in sorted(glob.glob(os.path.join(root, "**", "verifiers"),
                                        recursive=True)):
                if all(os.path.isdir(os.path.join(hit, p))
                       for p in ("invsqrt", "dqi", "dfthcsos")):
                    VERIFIERS = hit
                    break
            if VERIFIERS:
                break
        if not VERIFIERS:
            print("verifiers      : not found - skipped")
        else:
            COMP_ROOT = os.path.dirname(os.path.normpath(VERIFIERS))
            env = dict(os.environ)
            env["PYTHONPATH"] = os.pathsep.join(
                [os.path.join(COMP_ROOT, "kickmix")] +
                [p for p in env.get("PYTHONPATH", "").split(os.pathsep)
                 if p and os.path.normpath(p) != os.path.normpath(COMP_ROOT)])
            for pkg, script, args in (
                    ("invsqrt", "verify_submission_invsqrt.py",
                     [os.path.join(CIRCUITS, "invsqrt.kmb")]),
                    ("dqi", "verify_submission_dqi.py",
                     [os.path.join(CIRCUITS, "dqi.kmb")]),
                    ("dfthcsos", "verify.py", [CIRCUITS])):
                path = os.path.join(VERIFIERS, pkg, script)
                if not os.path.exists(path):
                    print("%-10s : verifier not found - skipped" % pkg)
                    continue
                print("=== %s ===" % pkg)
                proc = subprocess.run([sys.executable, path] + args,
                                      capture_output=True, text=True, env=env)
                try:
                    rep = json.loads(proc.stdout)
                    print("   status=%s score=%s" % (rep.get("status"),
                                                     rep.get("score")))
                    with open(os.path.join(WORK, "receipts",
                                           "%s-verify.json" % pkg), "w") as fh:
                        json.dump(rep, fh, indent=2, default=str)
                except Exception:
                    print("   did not return JSON:")
                    print(proc.stdout[-800:])
                    print(proc.stderr[-800:])
    except Exception as exc:
        print("optional verifier step failed (it never affects the archive):",
              exc)
else:
    print("official verifiers: SKIPPED (RUN_OFFICIAL_VERIFIERS = False)")

print()
print("UPLOAD THIS FILE: submission.zip")
print("Do not upload anything else.  Do not upload this notebook.")
